# Install Dependencies

In [ ]:
!pip install -q transformers==4.45.2
!pip install -q tokenizers==0.20.1
!pip install -q sentencepiece==0.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 139.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 181.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.2 MB/s eta 0:00:00


In [ ]:
!pip install -q arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 18.4 MB/s eta 0:00:00


# News Classification

In [ ]:
!unzip "/content/Sanad/Arabiya.zip" -d "/content/Sanad/Arabiya"
!unzip "/content/Sanad/Khaleej.zip" -d "/content/Sanad/Khaleej"
!unzip "/content/Sanad/Akhbarona.zip" -d "/content/Sanad/Akhbarona"

Streaming output truncated to the last 5000 lines.
  inflating: /content/Sanad/Akhbarona/Tech/02392.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02435.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02437.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02458.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02468.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02469.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02479.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02490.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02492.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02521.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02539.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02542.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02548.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02558.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02581.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02583.txt  
  inflating: /content/Sanad/Akhbarona/Tech/02585.txt  
  inflating: /

## Load dataset

In [ ]:
import os
import pandas as pd

root_dir = "Sanad"

data = []

for directory in os.listdir(root_dir):
    directory_path = os.path.join(root_dir, directory)

    # Make sure it is a directory
    if not os.path.isdir(directory_path):
        continue

    # Each subdirectory is a news class
    for news_class in os.listdir(directory_path):
        class_path = os.path.join(directory_path, news_class)

        if not os.path.isdir(class_path):
            continue

        # Read all txt files
        for filename in os.listdir(class_path):
            if filename.endswith(".txt"):
                file_path = os.path.join(class_path, filename)

                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()

                data.append({
                    "content": text,
                    "class": news_class,
                    "directory": directory
                })

df = pd.DataFrame(data)

print(df.shape[0])

195174


In [ ]:
df.columns

Index(['content', 'class', 'directory'], dtype='object')

In [ ]:
df.dropna(inplace = True, subset = 'content')

In [ ]:
test_samples = pd.read_excel('sampled_data-NewsClassification.xlsx')
test_samples.head(1)

,main directory,subdirectory,content
0,Khaleej,Culture,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...


In [ ]:
test_samples.shape[0]

1000

In [ ]:
data_filtered = df[~df["content"].isin(test_samples["content"])]
data_filtered .shape[0]

194149

In [ ]:
data_filtered.reset_index(drop = True, inplace = True)

# Split Data

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    data_filtered, test_size=0.3,
    random_state=42, stratify = data_filtered['class']
)

dev, test = train_test_split(
    temp, test_size=0.3,
    random_state=42, stratify = temp['class']
)

train['split'] = 'train'
dev['split'] = 'dev'
test['split'] = 'test'

split_data = pd.concat([train, dev, test])

In [ ]:
from datasets import Dataset, DatasetDict

# Create datasets for each split
news = DatasetDict({
    split: Dataset.from_pandas(
        split_data[split_data["split"] == split][["content", "class"]],
        preserve_index=False
    )
    for split in ["train", "dev", "test"]
})

## Preprocess

The next step is to load a T5 tokenizer to process `text` and `summary`:

In [ ]:
from transformers import AutoTokenizer

checkpoint = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
prefix = "تصنيف الخبر: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["content"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["class"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_news = news.map(preprocess_function, batched=True)

Map:   0%|          | 0/135904 [00:00<?, ? examples/s]

Map:   0%|          | 0/40771 [00:00<?, ? examples/s]

Map:   0%|          | 0/17474 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

## Evaluate

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_predictions = [
        p.strip()
        for p in decoded_predictions
    ]

    decoded_labels = [
        l.strip()
        for l in decoded_labels
    ]

    accuracy = accuracy_score(
        decoded_labels,
        decoded_predictions
    )

    f1 = f1_score(
        decoded_labels,
        decoded_predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

## Train

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="sentiment_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_news["train"],
    eval_dataset=tokenized_news["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.068900,0.066215,0.966545,0.963556
2,0.054300,0.051964,0.971843,0.969738
3,0.041000,0.049963,0.973707,0.971970
4,0.031000,0.049088,0.974811,0.973209


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control

TrainOutput(global_step=33976, training_loss=0.10515476081521197, metrics={'train_runtime': 13549.3238, 'train_samples_per_second': 40.121, 'train_steps_per_second': 2.508, 'total_flos': 8.322780340623606e+17, 'train_loss': 0.10515476081521197, 'epoch': 4.0})

## Inference

In [ ]:
text = 'لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتله بتنسيقه مع العصابة السيسية ودعمها فانتقم الله منه https://t.co/anjidHMCzK'

In [ ]:
from transformers import AutoTokenizer
dir = 'sentiment_model/checkpoint-33976'
tokenizer = AutoTokenizer.from_pretrained(dir)
inputs = tokenizer(text, return_tensors="pt").input_ids

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(dir)
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)

In [ ]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

'Politics'

In [ ]:
sentiment = []
for text in test_samples['content']:
  inputs = tokenizer(text, return_tensors="pt").input_ids
  outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)
  sentiment.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

Token indices sequence length is longer than the specified maximum sequence length for this model (1095 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
test_samples['Predicted Class'] = sentiment
test_samples.to_excel('Predicted Class by AraT5.xlsx', index = False)
test_samples.head(2)

,main directory,subdirectory,content,Predicted Class
0,Khaleej,Culture,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...,Culture
1,Khaleej,Culture,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...,Culture


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_samples['subdirectory'],
                            test_samples['Predicted Class'],
                            digits =4 ))

              precision    recall  f1-score   support

     Culture     0.9859    0.9333    0.9589       150
     Finance     0.9177    0.9667    0.9416       150
     Medical     0.9866    0.9800    0.9833       150
    Politics     0.9610    0.9867    0.9737       150
    Religion     0.9515    0.9800    0.9655       100
      Sports     0.9934    1.0000    0.9967       150
        Tech     0.9790    0.9333    0.9556       150

    accuracy                         0.9680      1000
   macro avg     0.9679    0.9686    0.9679      1000
weighted avg     0.9687    0.9680    0.9680      1000

